In [0]:
from pyspark.sql import SparkSession


In [0]:
%sql
create catalog telecom_catalog_assign;  

In [0]:
%sql
create schema telecom_catalog_assign.landing_zone;

In [0]:
%sql
create volume if not exists landing_vol;
drop volume if exists landing_vol;

In [0]:
%sql
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;


In [0]:
customer_csv= ''' 101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID '''

In [0]:
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer_csv.txt",customer_csv,True)

In [0]:
usage_tsv = ''' 101\t320\t1500\t20
 102\t120\t4000\t5
 103\t540\t600\t52
 104\t45\t200\t2
 105\t0\t0\t0 '''

In [0]:
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_tsv.txt",usage_tsv,True)

In [0]:
tower_logs_region1 = ''' 5001|101|TWR01|-80|2025-01-10 10:21:54
 5004|104|TWR05|-75|2025-01-10 11:01:12 '''

In [0]:
dbutils.fs.rm("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.logs.region1", True); dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.logs.region1",tower_logs_region1,True)


In [0]:
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_tsv.txt"))

**3. Spark Directory Read Use Cases**

In [0]:
df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/",header=False,inferSchema=True,sep="|").toDF("event_id","customer_id","tower_id","signal_strength","timestamp")

display(df)

**82.Demonstrate these 3 reads separately: Using pathGlobFilter Using list of paths in spark.read.csv([path1, path2]) Using .option("recursiveFileLookup","true")**

In [0]:
path1 = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/"
path2 = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/"

df2 = (
    spark.read
    .option("header","True")
    .option("inferSchema","True")
    .option("recursiveFileLookup", "true")
    .csv([path1, path2])
)

df2.show()

**4. Schema Inference, Header, and Separator**

**Try the Customer, Usage files with the option and options using read.csv and format function:
header=false, inferSchema=false
or
header=true, inferSchema=true**

In [0]:
df3=spark.read.options(header='false', inferSchema='True',sep=",").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")

display(df3)

###Write a note on What changed when we use header or inferSchema with true/false?

####true senario

**when we give header=true it will consider 1st column as header**

**when we give inferschema=true it will the datatypes in the give directories**

####false senario

**when we give header=false it will write the column with c0,c1,c2... when the files does not have the header**

**when we give inferschema=false it will define all the data as string type**

###How schema inference handled “abc” in age?

**it will consider as string**

**5. Column Renaming Usecases
Apply column names using string using toDF function for customer data
Apply column names and datatype using the schema function for usage data
Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data**

In [0]:
df5=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/").toDF("customer_id","cust_name","batch_id","city","payment_status")
display(df5)

In [0]:
df6=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/",inferSchema=True,sep="\t").toDF("cust_id","voice_mins","data_mb","tms_count")
display(df6)
df6.printSchema()

In [0]:
from pyspark.sql.types import StructType,StructField, StringType, LongType, IntegerType, TimestampType
cust_schema = StructType([
    StructField("event_id",LongType(),False),
    StructField("customer_id",IntegerType(),True),
    StructField("tower_id",StringType(),True),StructField("signal_strength",IntegerType(),True),
    StructField("timestamp",TimestampType(),True)])
df7 = spark.read.schema(cust_schema).option("ignoreLeadingWhiteSpace","true").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower.logs.region1",sep="|")

# CSV reads always mark columns nullable; re-create with the strict schema so event_id is NOT NULL
df7 = spark.createDataFrame(df7.collect(), cust_schema)

display(df7)
df7.printSchema()



###Write Operations (Data Conversion/Schema migration) – CSV Format Usecases

In [0]:
df3.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/output/customer/", mode="overwrite")
display(df3)

In [0]:
df6.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/",mode="Append")
display(df6)
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/_SUCCESS/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_tsv.txt/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/part-00000-tid-7052708231715533077-000dd7b6-8a4c-47d8-8d80-fbc7153139f0-146-1-c000.csv/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/_started_7229007937128335357/Volumes/telecom_catalog_assign/landing_zone/landing_vol/output//Volumes/telecom_catalog_assign/landing_zone/landing_vol/output//Volumes/telecom_catalog_assign/landing_zone/landing_vol/output/

In [0]:
df.write.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/output/tower/", mode="overwrite", header=True, sep="|")
display(df)